In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

metadata_df

,sample_title,treatment,upin,tissue,disease state
samples,,,,,
GSM5047691,pretreatment AML_3003486-N,pretreatment,3003486-N,bone marrow,AML
GSM5047692,pretreatment AML_3044356-N,pretreatment,3044356-N,bone marrow,AML
GSM5047693,pretreatment AML_3091926-N,pretreatment,3091926-N,bone marrow,AML
GSM5047694,pretreatment AML_3103988-N,pretreatment,3103988-N,bone marrow,AML
GSM5047695,pretreatment AML_3118298-N,pretreatment,3118298-N,bone marrow,AML
...,...,...,...,...,...
GSM5047767,pretreatment AML_3305912-N,pretreatment,3305912-N,bone marrow,AML
GSM5047768,pretreatment AML_3306954-N,pretreatment,3306954-N,bone marrow,AML
GSM5047769,pretreatment AML_3308122-N,pretreatment,3308122-N,bone marrow,AML


In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()

# https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE165656&format=file&file=GSE165656%5FClinical%5Fmetadata%2Exlsx
df_clinical_data = pd.read_excel('GSE165656_Clinical_metadata.xlsx', sheet_name=0)
df_clinical_data.to_csv('GSE165656_Clinical_metadata.csv', index=False)

df_combined = df_clinical_data.merge(
    df_combined,
    left_on='Patient_ID',
    right_on='upin',
    how='inner'
)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.to_csv(data_file_name, index=False)
df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_98566/3500072864.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,Patient_ID,Cluster_Definition_Post_FAB_Correction,Batches,Platform-RNASeq,Platform-FM1,Platform-RPPA,DX,FAB,WHO4Class,WHO.Narrow,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
0,3003486-N,Cluster_3,Batch1,RNASeq,FM1,RPPA,AML,M4,Not in Other,Not in Other,...,635,2,0,1,507,158,2,218,5,13
1,3044356-N,Cluster_2,Batch1,RNASeq,FM1,RPPA,AML,M4,Therapy Related,TX Rel-Chemo&XRT,...,4328,10,5,9,2222,495,76,2311,20,31
2,3091926-N,Cluster_1,Batch1,RNASeq,FM1,RPPA,AML,M4,Not in Other,Not in Other,...,11622,14,10,18,6236,1467,223,6381,39,193
3,3103988-N,Cluster_2,Batch1,RNASeq,FM1,RPPA,AML,M4,Not in Other,Not in Other,...,4913,5,3,4,3131,782,128,2174,12,36
4,3118298-N,Cluster_3,Batch1,RNASeq,FM1,RPPA,AML,M4,Not in Other,Not in Other,...,2450,6,3,2,1358,302,36,1029,5,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,3306954-N,Cluster_1,Batch3A,RNASeq,FM1,RPPA,AML,M4,Multilineage Dysplasia,PriorMDS-MPD,...,3014,4,4,3,1971,469,64,1392,8,16
77,3308122-N,Cluster_3,Batch3A,RNASeq,FM1,RPPA,AML,M1,Not in Other,Not in Other,...,5096,4,3,3,4099,1011,147,2609,10,48
78,3319790-N,Cluster_3,Batch3A,RNASeq,FM1,RPPA,AML,M5A,Not in Other,Not in Other,...,2286,1,2,2,1862,391,58,994,3,19
79,3282000-N,Cluster_2,Batch3A,RNASeq,FM1,RPPA,AML,M5,Not in Other,Not in Other,...,696,0,1,1,488,115,14,408,0,2


# Generate description

In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'GSE165656_Clinical_metadata.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
